
Project : Pentaho Log Intelligence

Layer   : Gold

Notebook: 03_Gold_Log_Catalina

Version : 1.0

Description:
Loads raw Pentaho log files from Unity Catalog Volume

into the Gold Delta table.

Author: Ernesto Felipe Garay Cervantes


#### Recibimiento de parametros

In [0]:
import json

dbutils.widgets.text("archivos_nuevos","")
archivos_nuevos = json.loads(dbutils.widgets.get("archivos_nuevos"))
print("====ARCHIVOS RECIBIDOS===")
for archivo in archivos_nuevos:
    print(archivo)

#### Configuracion

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    sha2,
    concat_ws,
    current_timestamp
)

In [0]:
CATALOG = "pentaho_logs"

SILVER_TABLE_CATALINA = "pentaho_logs.silver.silver_logs_catalina"

GOLD_TABLE_CATALINA= "pentaho_logs.gold.gold_logs_catalina"

#### Lectura  de tabla Bronze 

In [0]:
df_catalina_sl = spark.table(SILVER_TABLE_CATALINA).filter(col("file_name").isin(archivos_nuevos))

#display(df_catalina_sl.limit(20))

In [0]:
df_catalina_sl.printSchema()

#### Creación de identificador de Evento Log Catalina

In [0]:
from pyspark.sql.functions import (col,lit,sha2,concat_ws,current_timestamp)

df_gold_catalina = (df_catalina_sl.withColumn("event_id",sha2(concat_ws("||",col("file_path"),col("hora"),col("Nivel")),256))
                 .withColumn("source_type",lit("CATALINA"))
                  .withColumn("gold_timestamp",current_timestamp())
                 )



In [0]:
#display(
#df_gold_catalina.select(
 #         "event_id",
  #        "file_name",
   #       "application",
    #      "fecha",
     #     "hora",
      #    "Nivel",
       #   "mensaje"
# ).limit(20)
#)

In [0]:
df_gold_catalina = df_gold_catalina.select( 
           "event_id",
          "file_name",
          "application",
          "fecha",
          "hora",
          "Nivel",
          "mensaje"
 )

In [0]:
#display(df_gold_catalina.limit(20))


#### validación DATAFRAME

In [0]:
print(f"Registros Gold: {df_gold_catalina.count():,}")

In [0]:
from pyspark.sql.functions import col, sum, when

df_gold_catalina.select(
    sum(when(col("mensaje").isNull(), 1).otherwise(0)).alias("mensaje_evento_null"),
    sum(when(col("fecha").isNull(), 1).otherwise(0)).alias("fecha_null")
).show()


#### Creación Tabla Gold Catalina

In [0]:
GOLD_TABLE_CATALINA = "pentaho_logs.gold.gold_logs_catalina"
(
    df_gold_catalina.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_TABLE_CATALINA)
)

In [0]:
#display(spark.table(GOLD_TABLE_CATALINA).limit(20))